# Notebook 04: Model Inference & REST API

**Tim CC26-PSU164** — Fair Price Finder for Freelancers

---

## 📖 Latar Belakang

Model sudah selesai di-train (notebook 03), tapi **model tidak berguna sampai bisa dipakai user**. Notebook ini menjembatani gap antara "model artifact" dan "produk yang bisa di-consume oleh frontend".

### Dua Tujuan

1. **Inference Function (Main Quest #4):** Function yang terima input user → return prediksi
2. **REST API (Side Quest #5):** Wrap inference jadi HTTP endpoint untuk tim Web Developer

### Mengapa Inference di Notebook Terpisah?

Training butuh GPU + banyak library. Inference jauh lebih ringan:
- CPU sudah cukup
- Single sample prediction
- Minimal dependencies

**Production best practice:** training pipeline ≠ inference pipeline.

## ✅ Checklist Coverage

### Main Quest
- [x] **#4** Kode inference sederhana

### Side Quest
- [x] **#5** REST API mandiri pakai FastAPI

In [45]:
from pathlib import Path
import os


def find_project_root():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "data").exists():
            return candidate
    return current_dir


PROJECT_ROOT = find_project_root()

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\laragon\www\fair-price-finder


In [46]:
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = DATA_DIR / "models"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"

MODEL_PATH = MODELS_DIR / 'freelance_pricer_final.keras'
SCALER_PATH = PREPROCESSED_DIR / 'scaler.pkl'
FEATURE_NAMES_PATH = PREPROCESSED_DIR / 'feature_names.pkl'
METADATA_PATH = MODELS_DIR / 'model_metadata.json'
PREPROCESSING_INFO_PATH = PREPROCESSED_DIR / 'preprocessing_info.json'

print(f'📁 Data dir: {DATA_DIR}')
print(f'📁 Models dir: {MODELS_DIR}')
print(f'📁 Preprocessed dir: {PREPROCESSED_DIR}')
print(f'✅ Model exists: {MODEL_PATH.exists()}')
print(f'✅ Scaler exists: {SCALER_PATH.exists()}')
print(f'✅ Feature names exists: {FEATURE_NAMES_PATH.exists()}')
print(f'✅ Metadata exists: {METADATA_PATH.exists()}')
print(f'✅ Preprocessing info exists: {PREPROCESSING_INFO_PATH.exists()}')

📁 Data dir: C:\laragon\www\fair-price-finder\data
📁 Models dir: C:\laragon\www\fair-price-finder\data\models
📁 Preprocessed dir: C:\laragon\www\fair-price-finder\data\preprocessed
✅ Model exists: True
✅ Scaler exists: True
✅ Feature names exists: True
✅ Metadata exists: True
✅ Preprocessing info exists: True


## STEP 1: Re-define Custom Components

### Mengapa Harus Re-define?

Saat kita save model dengan custom Layer/Loss, TensorFlow **hanya save struktur**, bukan source code-nya.

Saat load model dengan `tf.keras.models.load_model()`, TF butuh **"resep"** custom class lewat parameter `custom_objects`. Jadi class definition harus tersedia di notebook ini.

**Alternative untuk production:** Save class definition di file `.py` terpisah, import dari sana. Untuk notebook-based capstone, redefine sudah cukup.

In [47]:
# IMPORT + RE-DEFINE CUSTOM COMPONENTS
import numpy as np
import pandas as pd
import pickle, json
import tensorflow as tf
from tensorflow.keras import layers

# Re-define ResidualDenseBlock (SAMA dengan notebook 02 versi FINAL)
class ResidualDenseBlock(layers.Layer):
    """
    Residual block dengan 2 Dense + LayerNormalization untuk stability.
    Harus sama persis dengan definisi di notebook 02.
    """
    def __init__(self, units, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.dense1 = layers.Dense(self.units, activation='relu')
        self.dropout = layers.Dropout(self.dropout_rate)
        self.dense2 = layers.Dense(input_shape[-1])
        self.layer_norm = layers.LayerNormalization()  # ← TAMBAH
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dropout(x, training=training)
        x = self.dense2(x)
        return self.layer_norm(inputs + x)  # ← UPDATE

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return config

print('✅ Custom components redefined (ResidualDenseBlock dengan LayerNorm)')

✅ Custom components redefined (ResidualDenseBlock dengan LayerNorm)


## STEP 2: Load Model & Artifacts

### Apa yang Perlu Di-load?

Untuk inference, kita butuh **4 komponen**:

| Component | Tujuan |
|---|---|
| **Model (.keras)** | Untuk predict |
| **Scaler (pkl)** | Transform input ke skala training |
| **Feature names (pkl)** | Memastikan urutan kolom benar |
| **Metadata (json)** | Clipping bounds untuk predictions |

**Tanpa salah satu = inference gagal atau hasilnya salah.**

In [59]:
# LOAD MODEL & ARTIFACTS
missing = [
    path for path in [MODEL_PATH, SCALER_PATH, FEATURE_NAMES_PATH]
    if not path.exists()
]
if missing:
    raise FileNotFoundError(
        'Missing required artifacts in data folders: '
        + ', '.join(str(path) for path in missing)
    )

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'ResidualDenseBlock': ResidualDenseBlock
    }
)

with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

with open(FEATURE_NAMES_PATH, 'rb') as f:
    feature_names = pickle.load(f)

if METADATA_PATH.exists():
    with open(METADATA_PATH, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
else:
    metadata = {
        'clipping_bounds': {
            'idr_p01': 50_000,
            'idr_p99': 4_500_000,
        },
        'test_metrics': None,
        'source': 'fallback_defaults',
    }
    print('⚠️ model_metadata.json not found, using fallback clipping bounds (Rp 50,000 - Rp 4,500,000)')

# Clipping bounds dari training
IDR_MIN = metadata['clipping_bounds']['idr_p01']
IDR_MAX = metadata['clipping_bounds']['idr_p99']

print(f'✅ Model loaded: {model.name}')
print(f'   Parameters: {model.count_params():,}')
print(f'   Features expected: {len(feature_names)}')
print(f'\n📊 Model performance (from training):')
if metadata.get('test_metrics'):
    print(f'   Test MAPE: {metadata["test_metrics"]["mape_pct"]:.2f}%')
    print(f'   Test R²:   {metadata["test_metrics"]["r2_log"]:.4f}')
else:
    print('   Test MAPE: N/A (metadata file not found)')
    print('   Test R²:   N/A (metadata file not found)')
print(f'\n📏 Clipping bounds: Rp {IDR_MIN:,.0f} - Rp {IDR_MAX:,.0f}')

✅ Model loaded: freelance_pricer_final
   Parameters: 11,297
   Features expected: 67

📊 Model performance (from training):
   Test MAPE: 56.13%
   Test R²:   0.4449

📏 Clipping bounds: Rp 50,000 - Rp 4,500,000


## STEP 3: Inference Function (Main Quest #4)

### Design Philosophy

Function `predict_price()` adalah **abstraction layer** antara model dan user. User tidak perlu tahu:
- Bagaimana feature di-encode (one-hot)
- Format internal model (log scale)
- Custom layer details
- Clipping logic

Mereka hanya provide **input yang natural**:

```python
user_input = {
    'kategori': 'Web dan Pemrograman',
    'platform': 'fastwork',
    'durasi_hari': 14,
    'skills': ['figma', 'ui ux design']
}
```

### Output: Range, Bukan Single Value

Project plan menyebut "rentang harga adil". Kami implement dengan:
- Model predict single value (median estimate)
- Post-processing: ±20% margin untuk min/max

**Why post-processing approach?**
- ✅ Tidak perlu retrain model
- ✅ Simple dan interpretable
- ⚠️ Range bukan hasil belajar model langsung

### Clipping Strategy

Predictions di-clip ke range training data (P1-P99) untuk:
- Mencegah extreme predictions (Rp 193 juta untuk listing Rp 350 ribu)
- Memastikan output dalam range yang masuk akal pasar

In [49]:
# INFERENCE FUNCTION (Main Quest #4)
def predict_price(user_input: dict, range_margin: float = 0.20) -> dict:
    """
    Predict harga adil untuk freelance project.

    Args:
        user_input: dict dengan keys:
            - kategori: str (e.g., 'Web dan Pemrograman')
            - platform: str (fastwork/sribu/projects)
            - durasi_hari: int
            - skills: list of str (e.g., ['figma', 'ui ux design'])
            - has_rating: bool (default True)
            - title_length: int (default 50)
            - desc_length: int (default 0)
            - has_urgency: bool (default False)
        range_margin: float, percentage for min/max range (default 20%)

    Returns:
        dict: predicted_price, price_min, price_max (semua dalam IDR)
    """
    # Initialize feature vector dengan 0
    features = {col: 0 for col in feature_names}

    # Set kategori
    kat_col = f'kategori_{user_input["kategori"]}'
    if kat_col in features:
        features[kat_col] = 1

    # Set platform
    plat_col = f'platform_{user_input["platform"]}'
    if plat_col in features:
        features[plat_col] = 1

    # Platform type flag
    features['is_service_based'] = 1 if user_input['platform'] in ['fastwork', 'sribu'] else 0

    # Numeric features
    durasi = user_input.get('durasi_hari', 7)
    features['durasi_hari'] = durasi
    features['log_durasi'] = np.log1p(durasi)
    features['has_rating'] = int(user_input.get('has_rating', True))
    features['title_length'] = user_input.get('title_length', 50)
    features['desc_length'] = user_input.get('desc_length', 0)
    features['has_description'] = 1 if features['desc_length'] > 0 else 0
    features['has_urgency'] = int(user_input.get('has_urgency', False))
    features['has_price_range'] = 0

    # Skills
    skills = user_input.get('skills', [])
    features['skill_count'] = len(skills)

    PREMIUM = {'machine learning', 'flutter'}
    features['has_premium_skill'] = 1 if any(s in PREMIUM for s in skills) else 0

    for skill in skills:
        skill_col = 'skill_' + skill.replace(' ', '_').replace('.', '').replace('/', '_')
        if skill_col in features:
            features[skill_col] = 1

    # Convert ke array sesuai urutan feature_names
    X = np.array([[features[col] for col in feature_names]])

    # Scale
    X_scaled = scaler.transform(X)

    # Predict
    log_pred = model.predict(X_scaled, verbose=0)[0][0]

    # Clip prediction dalam log range training
    log_pred_clipped = np.clip(log_pred, np.log1p(IDR_MIN), np.log1p(IDR_MAX))

    # Inverse transform ke IDR
    pred_idr = np.expm1(log_pred_clipped)

    # Apply range margin
    return {
        'predicted_price': int(pred_idr),
        'price_min': int(pred_idr * (1 - range_margin)),
        'price_max': int(pred_idr * (1 + range_margin)),
        'currency': 'IDR',
        'confidence': 'medium'
    }

print('✅ Inference function ready')

✅ Inference function ready


In [50]:
# AUTO-DETECT KATEGORI DARI SKILLS
SKILL_TO_CATEGORY = {
    # Web Dev & Programming
    'wordpress': 'Web dan Pemrograman',
    'react': 'Web dan Pemrograman',
    'laravel': 'Web dan Pemrograman',
    'flutter': 'Web dan Pemrograman',
    'python': 'Web dan Pemrograman',
    'html_css': 'Web dan Pemrograman',

    # Design
    'figma': 'Grafis & Desain',
    'photoshop': 'Grafis & Desain',
    'illustrator': 'Grafis & Desain',
    'canva': 'Grafis & Desain',
    'ui ux design': 'Grafis & Desain',
    'logo design': 'Grafis & Desain',
    'branding': 'Grafis & Desain',

    # Marketing
    'seo': 'Pemasaran Digital',
    'google ads': 'Pemasaran Digital',
    'tiktok ads': 'Pemasaran Digital',
    'instagram': 'Pemasaran Digital',
    'copywriting': 'Pemasaran Digital',

    # Writing
    'content writing': 'Penulisan & Penerjemahan',
    'translation': 'Penulisan & Penerjemahan',

    # Data & AI
    'machine learning': 'Web dan Pemrograman',
    'data analysis': 'Web dan Pemrograman',
    'excel': 'Web dan Pemrograman',

    # Video
    'video editing': 'Video & Animasi',
    'animation': 'Video & Animasi',
}

def detect_category(skills: list) -> str:
    """Auto-detect kategori paling cocok dari skills."""
    if not skills:
        return 'Web dan Pemrograman'  # Default

    # Count kategori dari skills
    cat_counts = {}
    for skill in skills:
        cat = SKILL_TO_CATEGORY.get(skill.lower())
        if cat:
            cat_counts[cat] = cat_counts.get(cat, 0) + 1

    # Return kategori paling sering
    if cat_counts:
        return max(cat_counts, key=cat_counts.get)
    return 'Web dan Pemrograman'  # Default fallback


# Test
print(detect_category(['figma', 'ui ux design']))  # → 'Grafis & Desain'
print(detect_category(['flutter', 'react']))        # → 'Web dan Pemrograman'
print(detect_category(['seo', 'google ads']))       # → 'Pemasaran Digital'

Grafis & Desain
Web dan Pemrograman
Pemasaran Digital


In [51]:
# SIMPLIFIED INFERENCE — User cuma input skills + durasi
def predict_price_simple(skills: list, durasi_hari: int, range_margin: float = 0.20) -> dict:
    """
    SIMPLIFIED inference untuk MVP.
    User cuma perlu kasih 2 input: skills dan durasi.
    Sisa parameter auto-default atau auto-detect.

    Args:
        skills: list of str (e.g., ['figma', 'ui ux design'])
        durasi_hari: int (e.g., 14)
        range_margin: float, optional (default 20%)

    Returns:
        dict: predicted_price, price_min, price_max (semua dalam IDR)
    """
    # Auto-detect kategori dari skills
    kategori = detect_category(skills)

    # Build full input dengan default
    user_input = {
        'kategori': kategori,
        'platform': 'fastwork',           # Default: platform paling umum
        'durasi_hari': durasi_hari,
        'skills': skills,
        'has_rating': True,               # Default: assume freelancer punya rating
        'title_length': 50,               # Default: rata-rata
        'desc_length': 0,                 # Default: no description
        'has_urgency': False              # Default: not urgent
    }

    # Pakai inference function full
    result = predict_price(user_input, range_margin=range_margin)

    # Tambah info kategori yang terdeteksi (untuk transparency)
    result['detected_category'] = kategori

    return result


# Test
print("Test 1: UI/UX Designer (14 hari)")
result = predict_price_simple(['figma', 'ui ux design'], 14)
print(f"  Detected: {result['detected_category']}")
print(f"  Predicted: Rp {result['predicted_price']:,}")
print(f"  Range: Rp {result['price_min']:,} - Rp {result['price_max']:,}")

print("\nTest 2: Flutter Dev (30 hari)")
result = predict_price_simple(['flutter'], 30)
print(f"  Detected: {result['detected_category']}")
print(f"  Predicted: Rp {result['predicted_price']:,}")
print(f"  Range: Rp {result['price_min']:,} - Rp {result['price_max']:,}")

Test 1: UI/UX Designer (14 hari)
  Detected: Grafis & Desain
  Predicted: Rp 203,797
  Range: Rp 163,037 - Rp 244,556

Test 2: Flutter Dev (30 hari)
  Detected: Web dan Pemrograman
  Predicted: Rp 261,187
  Range: Rp 208,949 - Rp 313,424


In [52]:
# TEST INFERENCE DENGAN BERBAGAI SCENARIO
test_cases = [
    {
        'name': 'Web Dev WordPress (7 hari, junior)',
        'input': {
            'kategori': 'Web dan Pemrograman',
            'platform': 'fastwork',
            'durasi_hari': 7,
            'skills': ['wordpress'],
            'has_rating': True,
            'title_length': 40
        }
    },
    {
        'name': 'Mobile Dev Flutter (30 hari, premium skill)',
        'input': {
            'kategori': 'Web dan Pemrograman',
            'platform': 'fastwork',
            'durasi_hari': 30,
            'skills': ['flutter'],
            'has_rating': True,
            'title_length': 60
        }
    },
    {
        'name': 'UI/UX Designer Figma (14 hari)',
        'input': {
            'kategori': 'Grafis & Desain',
            'platform': 'fastwork',
            'durasi_hari': 14,
            'skills': ['figma', 'ui ux design'],
            'has_rating': True,
            'title_length': 50
        }
    },
    {
        'name': 'Content Writer (3 hari, urgent)',
        'input': {
            'kategori': 'Penulisan & Penerjemahan',
            'platform': 'sribu',
            'durasi_hari': 3,
            'skills': ['content writing'],
            'has_rating': True,
            'has_urgency': True
        }
    },
]

print('='*70)
print('INFERENCE TEST CASES')
print('='*70)
for tc in test_cases:
    result = predict_price(tc['input'])
    print(f'\n🔹 {tc["name"]}')
    print(f'   Predicted: Rp {result["predicted_price"]:,}')
    print(f'   Range:     Rp {result["price_min"]:,} - Rp {result["price_max"]:,}')

INFERENCE TEST CASES

🔹 Web Dev WordPress (7 hari, junior)
   Predicted: Rp 216,862
   Range:     Rp 173,489 - Rp 260,234

🔹 Mobile Dev Flutter (30 hari, premium skill)
   Predicted: Rp 264,440
   Range:     Rp 211,552 - Rp 317,328

🔹 UI/UX Designer Figma (14 hari)
   Predicted: Rp 203,797
   Range:     Rp 163,037 - Rp 244,556

🔹 Content Writer (3 hari, urgent)
   Predicted: Rp 152,195
   Range:     Rp 121,756 - Rp 182,634


## STEP 4: REST API dengan FastAPI (Side Quest #5)

### Mengapa REST API?

Tim Frontend (React/HTML) **tidak bisa langsung call Python function** dari JavaScript. Mereka butuh HTTP endpoint untuk dapat prediksi.

**REST API = bridge** antara model (Python) dan frontend (web/mobile).

### Mengapa FastAPI (Bukan Flask)?

| Framework | Pros | Cons |
|---|---|---|
| Flask | Mature, banyak tutorial | Manual validation, lebih lambat |
| **FastAPI** ✅ | **Auto docs (Swagger), type hints, async, fast** | Lebih baru |

FastAPI dipilih karena:
- **Auto-generated Swagger docs** (mudah testing untuk Web Developer)
- **Type validation** via Pydantic
- **Async by default**
- Industry trend

### Mengapa Pakai Pydantic?

Pydantic model untuk **validate request payload** secara otomatis:

```python
class PredictionRequest(BaseModel):
    kategori: str        # Required, must be string
    durasi_hari: int     # Required, must be int
```

FastAPI akan auto-reject request invalid dengan error message yang jelas.

### Mengapa Pakai ngrok?

Colab jalan di Google's private network. Untuk expose API ke internet (agar Web Developer bisa test dari laptop mereka), butuh **tunnel service** seperti ngrok.

> *Production deployment: deploy ke Railway, Render, atau cloud VM. ngrok cukup untuk demo.*

In [53]:
# INSTALL FASTAPI & DEPENDENCIES
!pip install fastapi uvicorn pyngrok nest-asyncio -q
print('✅ Installed')

✅ Installed


In [54]:
# HELPER FUNCTIONS untuk Simplified Inference
import numpy as np

# Mapping skill ke kategori
SKILL_TO_CATEGORY = {
    # Web Dev & Programming
    'wordpress': 'Web dan Pemrograman',
    'react': 'Web dan Pemrograman',
    'laravel': 'Web dan Pemrograman',
    'flutter': 'Web dan Pemrograman',
    'python': 'Web dan Pemrograman',
    'html_css': 'Web dan Pemrograman',

    # Design
    'figma': 'Grafis & Desain',
    'photoshop': 'Grafis & Desain',
    'illustrator': 'Grafis & Desain',
    'canva': 'Grafis & Desain',
    'ui ux design': 'Grafis & Desain',
    'logo design': 'Grafis & Desain',
    'branding': 'Grafis & Desain',

    # Marketing
    'seo': 'Pemasaran Digital',
    'google ads': 'Pemasaran Digital',
    'tiktok ads': 'Pemasaran Digital',
    'instagram': 'Pemasaran Digital',
    'copywriting': 'Pemasaran Digital',

    # Writing
    'content writing': 'Penulisan & Penerjemahan',
    'translation': 'Penulisan & Penerjemahan',

    # Data & AI
    'machine learning': 'Web dan Pemrograman',
    'data analysis': 'Web dan Pemrograman',
    'excel': 'Web dan Pemrograman',

    # Video
    'video editing': 'Video & Animasi',
    'animation': 'Video & Animasi',
}

def detect_category(skills):
    """Auto-detect kategori paling cocok dari skills."""
    if not skills:
        return 'Web dan Pemrograman'
    cat_counts = {}
    for skill in skills:
        cat = SKILL_TO_CATEGORY.get(skill.lower())
        if cat:
            cat_counts[cat] = cat_counts.get(cat, 0) + 1
    if cat_counts:
        return max(cat_counts, key=cat_counts.get)
    return 'Web dan Pemrograman'


def predict_price(user_input, range_margin=0.20):
    """Full inference function (semua parameter)."""
    features = {col: 0 for col in feature_names}

    # Set kategori & platform
    kat_col = f'kategori_{user_input["kategori"]}'
    if kat_col in features:
        features[kat_col] = 1

    plat_col = f'platform_{user_input["platform"]}'
    if plat_col in features:
        features[plat_col] = 1

    features['is_service_based'] = 1 if user_input['platform'] in ['fastwork', 'sribu'] else 0

    # Numeric features
    durasi = user_input.get('durasi_hari', 7)
    features['durasi_hari'] = durasi
    features['log_durasi'] = np.log1p(durasi)
    features['has_rating'] = int(user_input.get('has_rating', True))
    features['title_length'] = user_input.get('title_length', 50)
    features['desc_length'] = user_input.get('desc_length', 0)
    features['has_description'] = 1 if features['desc_length'] > 0 else 0
    features['has_urgency'] = int(user_input.get('has_urgency', False))
    features['has_price_range'] = 0

    # Skills
    skills = user_input.get('skills', [])
    features['skill_count'] = len(skills)
    PREMIUM = {'machine learning', 'flutter'}
    features['has_premium_skill'] = 1 if any(s in PREMIUM for s in skills) else 0

    for skill in skills:
        skill_col = 'skill_' + skill.replace(' ', '_').replace('.', '').replace('/', '_')
        if skill_col in features:
            features[skill_col] = 1

    # Predict
    X = np.array([[features[col] for col in feature_names]])
    X_scaled = scaler.transform(X)
    log_pred = model.predict(X_scaled, verbose=0)[0][0]
    log_pred_clipped = np.clip(log_pred, np.log1p(IDR_MIN), np.log1p(IDR_MAX))
    pred_idr = np.expm1(log_pred_clipped)

    return {
        'predicted_price': int(pred_idr),
        'price_min': int(pred_idr * (1 - range_margin)),
        'price_max': int(pred_idr * (1 + range_margin)),
        'currency': 'IDR'
    }


def predict_price_simple(skills, durasi_hari, range_margin=0.20):
    """
    SIMPLIFIED inference untuk MVP.
    User cuma input 2 hal: skills + durasi. Sisa auto-default.
    """
    kategori = detect_category(skills)

    user_input = {
        'kategori': kategori,
        'platform': 'fastwork',
        'durasi_hari': durasi_hari,
        'skills': skills,
        'has_rating': True,
        'title_length': 50,
        'desc_length': 0,
        'has_urgency': False
    }

    result = predict_price(user_input, range_margin=range_margin)
    result['detected_category'] = kategori
    return result


# Quick test
print('✅ Helper functions ready')
print('\n🧪 Quick test:')
test = predict_price_simple(['figma', 'ui ux design'], 14)
print(f"   Skills: figma + ui ux design, 14 hari")
print(f"   → Kategori: {test['detected_category']}")
print(f"   → Predicted: Rp {test['predicted_price']:,}")
print(f"   → Range: Rp {test['price_min']:,} - Rp {test['price_max']:,}")

✅ Helper functions ready

🧪 Quick test:
   Skills: figma + ui ux design, 14 hari
   → Kategori: Grafis & Desain
   → Predicted: Rp 203,797
   → Range: Rp 163,037 - Rp 244,556


In [55]:
# FASTAPI APP DEFINITION
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List

app = FastAPI(
    title='Fair Price Finder API',
    description='REST API untuk prediksi harga adil freelance project Indonesia',
    version='1.0',
    contact={'name': 'Tim CC26-PSU164'}
)

# Allow CORS untuk frontend integration
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ----- Schemas -----
class SimplePredictionRequest(BaseModel):
    """User cuma kasih skills + durasi (MVP)."""
    skills: List[str] = Field(..., min_items=1, description='List skills')
    durasi_hari: int = Field(..., ge=1, le=365, description='Durasi pengerjaan')

    class Config:
        json_schema_extra = {
            'example': {
                'skills': ['figma', 'ui ux design'],
                'durasi_hari': 14
            }
        }

class SimplePredictionResponse(BaseModel):
    predicted_price: int
    price_min: int
    price_max: int
    detected_category: str
    currency: str = 'IDR'

class FullPredictionRequest(BaseModel):
    """Full input untuk advanced use case."""
    kategori: str
    platform: str = 'fastwork'
    durasi_hari: int = Field(..., ge=1, le=365)
    skills: List[str] = []
    has_rating: bool = True
    title_length: int = 50
    desc_length: int = 0
    has_urgency: bool = False

# ----- Endpoints -----
@app.get('/')
def root():
    return {
        'service': 'Fair Price Finder API',
        'version': '1.0',
        'status': 'online',
        'team': 'CC26-PSU164',
        'docs': '/docs'
    }

@app.get('/health')
def health():
    return {'status': 'healthy', 'model_loaded': True}

@app.post('/predict', response_model=SimplePredictionResponse)
def predict(request: SimplePredictionRequest):
    """
    MAIN ENDPOINT — Simplified prediction untuk frontend.
    User cuma kasih skills + durasi.
    """
    try:
        result = predict_price_simple(
            skills=request.skills,
            durasi_hari=request.durasi_hari
        )
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/predict_advanced')
def predict_advanced(request: FullPredictionRequest):
    """Advanced endpoint untuk full control input."""
    try:
        return predict_price(request.dict())
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get('/skills')
def get_skills():
    """List valid skills untuk frontend dropdown."""
    skills = [c.replace('skill_', '').replace('_', ' ')
              for c in feature_names if c.startswith('skill_')]
    return {'skills': sorted(skills), 'total': len(skills)}

@app.get('/categories')
def get_categories():
    cats = [c.replace('kategori_', '') for c in feature_names if c.startswith('kategori_')]
    return {'categories': cats}

print('✅ FastAPI app defined')
print(f'   Endpoints: /, /health, /predict, /predict_advanced, /skills, /categories')

✅ FastAPI app defined
   Endpoints: /, /health, /predict, /predict_advanced, /skills, /categories


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14048\1389820092.py:25: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  skills: List[str] = Field(..., min_items=1, description='List skills')
C:\Users\ASUS\AppData\Local\Temp\ipykernel_14048\1389820092.py:23: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class SimplePredictionRequest(BaseModel):


In [57]:
# START SERVER + NGROK (Local Jupyter compatible — tanpa google.colab)
import uvicorn
import asyncio
from pyngrok import ngrok
from threading import Thread
import time
import requests
import os

# Stop tunnel lama (kalau ada)
try:
    ngrok.kill()
except:
    pass

# Setup ngrok token — coba dari environment variable
print('🔑 Attempting to get ngrok token...')
NGROK_AUTHTOKEN = os.getenv('NGROK_AUTHTOKEN', None)

if not NGROK_AUTHTOKEN:
    print('⚠️  NGROK_AUTHTOKEN not found in environment variables.')
    print('📌 Cara setup:')
    print('   1. Get token dari: https://dashboard.ngrok.com/auth/your-authtoken')
    print('   2. Set environment variable: $env:NGROK_AUTHTOKEN="your_token"')
    print('   3. Re-run sel ini')
    print('\n💡 Alternatif: Skip ngrok tunnel, pakai local API saja')
    print('   Server akan berjalan di: http://localhost:8000')
    print('   Swagger docs di: http://localhost:8000/docs')
    use_ngrok = False
else:
    print(f'✅ NGROK_AUTHTOKEN found')
    try:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
        use_ngrok = True
    except Exception as e:
        print(f'⚠️  Failed to set ngrok token: {e}')
        use_ngrok = False

# STEP 1: Start uvicorn server di background thread
print('\n🚀 Step 1: Starting FastAPI server...')

config = uvicorn.Config(
    app=app,
    host='0.0.0.0',
    port=8000,
    log_level='warning'
)
server = uvicorn.Server(config)

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# STEP 2: Wait for server to be ready
print('⏳ Step 2: Waiting 10 seconds for server initialization...')
time.sleep(10)

# STEP 3: Verify server running locally
print('🔍 Step 3: Verifying server...')
try:
    test = requests.get('http://localhost:8000/health', timeout=5)
    if test.status_code == 200:
        print(f'   ✅ Server is running: {test.json()}')
    else:
        print(f'   ⚠️ Server responded with status {test.status_code}')
        raise SystemExit('Server not healthy')
except Exception as e:
    print(f'   ❌ Server NOT responding: {e}')
    print('   → Restart runtime dan re-run dari awal')
    raise SystemExit('Server failed')

# STEP 4: Create ngrok tunnel (jika token tersedia)
print('\n' + '='*60)
if use_ngrok:
    print('🌐 Step 4: Creating ngrok tunnel...')
    try:
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
        print(f'✅ ngrok tunnel created!')
        print(f'🌐 Public URL:   {public_url}')
        print(f'📚 Swagger Docs: {public_url}/docs')
        print(f'\n🎯 Endpoint untuk Frontend:')
        print(f'   POST {public_url}/predict')
        print(f'   GET  {public_url}/skills')
        print(f'   GET  {public_url}/categories')
        print(f'\n💡 Share URL ini ke tim Web Developer!')
    except Exception as e:
        print(f'❌ Failed to create ngrok tunnel: {e}')
        print('⚠️  Using local API saja')
        use_ngrok = False
        public_url = None
else:
    print('🔗 Using LOCAL API saja (tanpa ngrok tunnel)')
    public_url = None

print('='*60)
print(f'✅ API READY!')
print('='*60)
if public_url:
    print(f'🌐 Public URL: {public_url}')
else:
    print(f'🌐 Local URL:  http://localhost:8000')
print(f'📚 Swagger:   http://localhost:8000/docs')

🔑 Attempting to get ngrok token...
⚠️  NGROK_AUTHTOKEN not found in environment variables.
📌 Cara setup:
   1. Get token dari: https://dashboard.ngrok.com/auth/your-authtoken
   2. Set environment variable: $env:NGROK_AUTHTOKEN="your_token"
   3. Re-run sel ini

💡 Alternatif: Skip ngrok tunnel, pakai local API saja
   Server akan berjalan di: http://localhost:8000
   Swagger docs di: http://localhost:8000/docs

🚀 Step 1: Starting FastAPI server...
⏳ Step 2: Waiting 10 seconds for server initialization...
🔍 Step 3: Verifying server...
   ✅ Server is running: {'status': 'healthy', 'model_loaded': True}

🔗 Using LOCAL API saja (tanpa ngrok tunnel)
✅ API READY!
🌐 Local URL:  http://localhost:8000
📚 Swagger:   http://localhost:8000/docs


In [58]:
# TEST API ENDPOINTS
import requests
import json
import time

time.sleep(2)  # Wait sebentar biar server ready

# Determine base URL
if 'public_url' in locals() and public_url:
    base_url = public_url
    print(f'🌐 Testing ngrok tunnel: {base_url}')
else:
    base_url = 'http://localhost:8000'
    print(f'🌐 Testing local server: {base_url}')

print('='*60)
print('🧪 1. HEALTH CHECK')
print('='*60)
try:
    r = requests.get(f'{base_url}/health', timeout=10)
    print(f'Status: {r.status_code}')
    print(f'Response: {json.dumps(r.json(), indent=2)}')
except Exception as e:
    print(f'❌ Error: {e}')

print('\n' + '='*60)
print('🧪 2. GET SKILLS LIST')
print('='*60)
try:
    r = requests.get(f'{base_url}/skills', timeout=10)
    data = r.json()
    print(f'Status: {r.status_code}')
    print(f'Total skills: {data["total"]}')
    print(f'Sample (first 10): {data["skills"][:10]}')
except Exception as e:
    print(f'❌ Error: {e}')

print('\n' + '='*60)
print('🧪 3. PREDICT (SIMPLIFIED — Skills + Durasi)')
print('='*60)

test_cases = [
    {'skills': ['figma', 'ui ux design'], 'durasi_hari': 14},
    {'skills': ['flutter'], 'durasi_hari': 30},
    {'skills': ['wordpress'], 'durasi_hari': 7},
    {'skills': ['content writing'], 'durasi_hari': 3},
]

for payload in test_cases:
    try:
        r = requests.post(f'{base_url}/predict', json=payload, timeout=10)
        result = r.json()
        print(f"\n📥 Input: {payload}")
        print(f"📤 Output:")
        print(f"   Kategori:  {result['detected_category']}")
        print(f"   Predicted: Rp {result['predicted_price']:,}")
        print(f"   Range:     Rp {result['price_min']:,} - Rp {result['price_max']:,}")
    except Exception as e:
        print(f"❌ Error predicting: {e}")

print('\n' + '='*60)
print('✅ ALL TESTS DONE! API ready untuk handover ke Web Developer')
print('='*60)

🌐 Testing local server: http://localhost:8000
🧪 1. HEALTH CHECK
Status: 200
Response: {
  "status": "healthy",
  "model_loaded": true
}

🧪 2. GET SKILLS LIST
Status: 200
Total skills: 46
Sample (first 10): ['3d modeling', 'adobe xd', 'after effects', 'animation', 'branding', 'canva', 'content writing', 'copywriting', 'count', 'data analysis']

🧪 3. PREDICT (SIMPLIFIED — Skills + Durasi)

📥 Input: {'skills': ['figma', 'ui ux design'], 'durasi_hari': 14}
📤 Output:
   Kategori:  Grafis & Desain
   Predicted: Rp 203,797
   Range:     Rp 163,037 - Rp 244,556

📥 Input: {'skills': ['flutter'], 'durasi_hari': 30}
📤 Output:
   Kategori:  Web dan Pemrograman
   Predicted: Rp 261,187
   Range:     Rp 208,949 - Rp 313,424

📥 Input: {'skills': ['wordpress'], 'durasi_hari': 7}
📤 Output:
   Kategori:  Web dan Pemrograman
   Predicted: Rp 218,208
   Range:     Rp 174,566 - Rp 261,850

📥 Input: {'skills': ['content writing'], 'durasi_hari': 3}
📤 Output:
   Kategori:  Penulisan & Penerjemahan
   Predict

## STEP 5: Handover ke Tim Web Developer

### Informasi Yang Harus Di-share

```markdown
## Fair Price Finder API — Integration Guide

**Base URL:** [URL dari ngrok di cell sebelumnya]
**Documentation:** {BASE_URL}/docs (interactive Swagger UI)

### Main Endpoint
POST /predict
Content-Type: application/json

### Request Body
{
    "kategori": "Web dan Pemrograman",
    "platform": "fastwork",
    "durasi_hari": 14,
    "skills": ["figma", "ui ux design"],
    "has_rating": true,
    "title_length": 50,
    "has_urgency": false
}

### Response
{
    "predicted_price": 850000,
    "price_min": 680000,
    "price_max": 1020000,
    "currency": "IDR",
    "confidence": "medium"
}

### Helper Endpoints (untuk Frontend dropdown/autocomplete)
- GET /categories  → list valid kategori
- GET /skills      → list valid skills
- GET /platforms   → list valid platforms
```

### ⚠️ Catatan Penting

1. **ngrok URL tidak permanent** — restart Colab = URL baru. Untuk production deploy ke Railway/Render.
2. **Free tier ngrok** punya rate limit. Untuk demo capstone cukup, untuk production butuh paid.
3. **CORS belum di-setup** — kalau Frontend dari domain berbeda, mungkin perlu tambah CORS middleware.

### Sample CORS Setup (Kalau Diperlukan)

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Untuk demo. Production: specific domain
    allow_methods=["*"],
    allow_headers=["*"],
)
```

## STEP 6: Deployment ke Production (Bonus)

### Mengapa Deploy?

Notebook 03 menjalankan API via ngrok di Colab — bagus untuk development,
tapi **tidak production-ready**:
- URL berubah saat Colab restart
- API mati saat Colab tutup
- Tidak bisa always-on

### Solusi: Hugging Face Spaces

Kami deploy code FastAPI yang sama (di-extract ke `app.py`) ke
Hugging Face Spaces untuk:
- ✅ URL permanen
- ✅ Always-on 24/7
- ✅ Free hosting

### Production URL:
**https://yourname-fair-price-finder.hf.space**

### Documentation (Swagger UI):
https://yourname-fair-price-finder.hf.space/docs

### Files yang Di-deploy:
| File | Tujuan |
|---|---|
| `app.py` | FastAPI code (extract dari Notebook 03 Cell 15) |
| `requirements.txt` | Dependencies |
| `Dockerfile` | Container config |
| `freelance_pricer_final.keras` | Model dari Notebook 02 |
| `scaler.pkl`, `feature_names.pkl` | Preprocessing artifacts |

### Verifikasi: